# PyTorch TrainJob from Feast → MinIO

This is a copy of `pytorch_trainjob_from_feast.ipynb` with one key change:
**the trained model is uploaded directly to MinIO (S3) instead of a PVC.**

This eliminates PVC multi-attach issues and lets KServe pull the model from `s3://models/fraud-detector/`.

Changes from the original:
- Training function has MinIO params (`minio_endpoint`, `minio_access_key`, etc.)
- After saving locally in `/tmp`, rank 0 uploads to MinIO
- `output_dir` changed from `/mnt/models` to `/tmp/model_output` (no PVC needed)
- `boto3` added to `packages_to_install`

## 1) Install dependencies

In [ ]:
!pip install -q kubeflow
!pip install -q torch pandas scikit-learn pyarrow boto3

## 2) Define training function (with MinIO upload)

Identical to the original `train_fraud_from_feast` except:
- Accepts MinIO connection params
- After training, rank 0 uploads model + metrics to MinIO
- No PVC mount needed for model output

In [ ]:
def train_fraud_from_feast(
    num_epochs=10,
    batch_size=512,
    lr=1e-3,
    hidden_dim=64,
    training_data_path="/mnt/data/feast_training.parquet",
    metadata_path="/mnt/data/feast_training_metadata.json",
    output_dir="/tmp/model_output",
    minio_endpoint="http://minio-service.kubeflow.svc.cluster.local:9000",
    minio_access_key="minio",
    minio_secret_key="minio123",
    minio_bucket="models",
    minio_model_prefix="fraud-detector",
):
    import json
    import os
    import random

    if isinstance(num_epochs, dict):
        cfg = num_epochs
        num_epochs = cfg.get('num_epochs', 10)
        batch_size = cfg.get('batch_size', 512)
        lr = cfg.get('lr', 1e-3)
        hidden_dim = cfg.get('hidden_dim', 64)
        training_data_path = cfg.get('training_data_path', training_data_path)
        metadata_path = cfg.get('metadata_path', metadata_path)
        output_dir = cfg.get('output_dir', output_dir)
        minio_endpoint = cfg.get('minio_endpoint', minio_endpoint)
        minio_access_key = cfg.get('minio_access_key', minio_access_key)
        minio_secret_key = cfg.get('minio_secret_key', minio_secret_key)
        minio_bucket = cfg.get('minio_bucket', minio_bucket)
        minio_model_prefix = cfg.get('minio_model_prefix', minio_model_prefix)

    import numpy as np
    import pandas as pd
    import torch
    import torch.distributed as dist
    from sklearn.metrics import roc_auc_score
    from torch import nn
    from torch.utils.data import DataLoader, Dataset, DistributedSampler

    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)

    class TabularDataset(Dataset):
        def __init__(self, frame, feature_cols, label_col):
            self.x = torch.tensor(frame[feature_cols].values, dtype=torch.float32)
            self.y = torch.tensor(frame[label_col].values, dtype=torch.float32).unsqueeze(1)

        def __len__(self):
            return len(self.x)

        def __getitem__(self, idx):
            return self.x[idx], self.y[idx]

    class FraudMLP(nn.Module):
        def __init__(self, input_dim, hidden):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, hidden),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(hidden, hidden // 2),
                nn.ReLU(),
                nn.Linear(hidden // 2, 1),
            )

        def forward(self, x):
            return self.net(x)

    if not os.path.exists(training_data_path):
        raise FileNotFoundError(
            f"Missing training dataset at {training_data_path}. Run feast_data_prep.ipynb first."
        )

    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    feature_cols = metadata["feature_columns"]
    label_col = metadata["label_column"]

    df = pd.read_parquet(training_data_path)
    train_df = df[df["split"] == "train"].copy()
    val_df = df[df["split"] == "val"].copy()

    if train_df.empty or val_df.empty:
        raise RuntimeError("Train/val splits are empty. Rebuild data artifact from feast_data_prep.ipynb.")

    world_size = int(os.getenv("WORLD_SIZE", "1"))
    rank = int(os.getenv("RANK", "0"))
    local_rank = int(os.getenv("LOCAL_RANK", "0"))

    distributed = world_size > 1
    if distributed:
        backend = "nccl" if torch.cuda.is_available() else "gloo"
        dist.init_process_group(backend=backend)

    if torch.cuda.is_available():
        device = torch.device(f"cuda:{local_rank}")
        torch.cuda.set_device(device)
    else:
        device = torch.device("cpu")

    model = FraudMLP(input_dim=len(feature_cols), hidden=hidden_dim).to(device)
    if distributed:
        model = nn.parallel.DistributedDataParallel(
            model,
            device_ids=[local_rank] if torch.cuda.is_available() else None,
        )

    train_dataset = TabularDataset(train_df, feature_cols, label_col)
    val_dataset = TabularDataset(val_df, feature_cols, label_col)

    train_sampler = DistributedSampler(train_dataset) if distributed else None
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=train_sampler,
        shuffle=(train_sampler is None),
    )
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(num_epochs):
        model.train()
        if train_sampler is not None:
            train_sampler.set_epoch(epoch)

        running_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        epoch_loss = running_loss / max(len(train_loader), 1)

        model.eval()
        val_targets = []
        val_scores = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                logits = model(xb)
                probs = torch.sigmoid(logits).cpu().numpy().reshape(-1)
                val_scores.extend(probs.tolist())
                val_targets.extend(yb.numpy().reshape(-1).tolist())

        if len(set(int(v) for v in val_targets)) >= 2:
            val_auc = roc_auc_score(val_targets, val_scores)
        else:
            val_auc = float("nan")

        if rank == 0:
            print(
                f"Epoch {epoch + 1}/{num_epochs} | loss={epoch_loss:.4f} | val_auc={val_auc:.4f}"
            )

    if rank == 0:
        os.makedirs(output_dir, exist_ok=True)
        model_to_save = model.module if hasattr(model, "module") else model
        model_path = os.path.join(output_dir, "fraud_mlp_state_dict.pt")
        metrics_path = os.path.join(output_dir, "metrics.json")

        torch.save(
            {
                "model_state_dict": model_to_save.state_dict(),
                "feature_columns": feature_cols,
                "label_column": label_col,
                "hidden_dim": hidden_dim,
            },
            model_path,
        )

        with open(metrics_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "val_auc": float(val_auc),
                    "epochs": int(num_epochs),
                    "train_rows": int(len(train_df)),
                    "val_rows": int(len(val_df)),
                },
                f,
                indent=2,
            )

        print(f"Saved model artifact: {model_path}")
        print(f"Saved metrics artifact: {metrics_path}")

        import boto3
        from botocore.client import Config as BotoConfig

        s3 = boto3.client(
            "s3",
            endpoint_url=minio_endpoint,
            aws_access_key_id=minio_access_key,
            aws_secret_access_key=minio_secret_key,
            config=BotoConfig(signature_version="s3v4"),
            region_name="us-east-1",
        )
        try:
            s3.create_bucket(Bucket=minio_bucket)
        except Exception:
            pass

        s3.upload_file(model_path, minio_bucket, f"{minio_model_prefix}/fraud_mlp_state_dict.pt")
        s3.upload_file(metrics_path, minio_bucket, f"{minio_model_prefix}/metrics.json")
        print(f"Uploaded to MinIO: s3://{minio_bucket}/{minio_model_prefix}/fraud_mlp_state_dict.pt")
        print(f"Uploaded to MinIO: s3://{minio_bucket}/{minio_model_prefix}/metrics.json")

    if distributed:
        dist.barrier()
        dist.destroy_process_group()

## 3) Local run (fast validation)

Same as original but with `output_dir=/tmp/model_output` and `boto3` in packages.

In [ ]:
from kubeflow.trainer import CustomTrainer, LocalProcessBackendConfig, TrainerClient

local_client = TrainerClient(backend_config=LocalProcessBackendConfig(cleanup_venv=True))

torch_runtime = None
for rt in local_client.list_runtimes():
    if rt.name == "torch-distributed":
        torch_runtime = rt
        break

if torch_runtime is None:
    raise RuntimeError("Local runtime 'torch-distributed' not found")

local_job_name = local_client.train(
    trainer=CustomTrainer(
        func=train_fraud_from_feast,
        func_args={
            "num_epochs": 2,
            "batch_size": 256,
            "lr": 1e-3,
            "hidden_dim": 64,
            "training_data_path": "/mnt/data/feast_training.parquet",
            "metadata_path": "/mnt/data/feast_training_metadata.json",
            "output_dir": "/tmp/model_output",
            "minio_endpoint": "http://minio-service.kubeflow.svc.cluster.local:9000",
            "minio_access_key": "minio",
            "minio_secret_key": "minio123",
            "minio_bucket": "models",
            "minio_model_prefix": "fraud-detector",
        },
        packages_to_install=["torch", "pandas", "scikit-learn", "pyarrow", "boto3"],
    ),
    runtime=torch_runtime,
)

print(f"Local job: {local_job_name}")
for line in local_client.get_job_logs(local_job_name, follow=True):
    print(line, end="")

## 4) Submit distributed TrainJob to Kubernetes

Same function, same MinIO params. Model goes directly to MinIO — no PVC needed for output.

In [ ]:
from kubeflow.trainer import CustomTrainer, TrainerClient

client = TrainerClient()

job_name = client.train(
    trainer=CustomTrainer(
        func=train_fraud_from_feast,
        func_args={
            "num_epochs": 10,
            "batch_size": 512,
            "lr": 1e-3,
            "hidden_dim": 64,
            "training_data_path": "/mnt/data/feast_training.parquet",
            "metadata_path": "/mnt/data/feast_training_metadata.json",
            "output_dir": "/tmp/model_output",
            "minio_endpoint": "http://minio-service.kubeflow.svc.cluster.local:9000",
            "minio_access_key": "minio",
            "minio_secret_key": "minio123",
            "minio_bucket": "models",
            "minio_model_prefix": "fraud-detector",
        },
        num_nodes=2,
        resources_per_node={
            "cpu": "2",
            "memory": "4Gi",
        },
        packages_to_install=["torch", "pandas", "scikit-learn", "pyarrow", "boto3"],
    ),
    runtime="torch-distributed",
)

print(f"Submitted TrainJob: {job_name}")

## 5) Monitor TrainJob and logs

In [ ]:
client.wait_for_job_status(name=job_name, status={"Running"}, timeout=300)
print(f"{job_name} is running. Streaming logs:")
for line in client.get_job_logs(job_name, follow=True):
    print(line, end="")

In [ ]:
client.wait_for_job_status(name=job_name, timeout=20)
print("TrainJob completed.")

## 6) Verify model in MinIO

In [ ]:
import boto3
from botocore.client import Config

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio-service.kubeflow.svc.cluster.local:9000",
    aws_access_key_id="minio",
    aws_secret_access_key="minio123",
    config=Config(signature_version="s3v4"),
    region_name="us-east-1",
)

print("Files in s3://models/fraud-detector/:")
objs = s3.list_objects_v2(Bucket="models", Prefix="fraud-detector/")
for obj in objs.get("Contents", []):
    print(f"  {obj['Key']} ({obj['Size']:,} bytes, modified: {obj['LastModified']})")